In [1]:
import numpy as np
import pandas as pd

In [22]:
import plotly.io as pio
pio.renderers.default = "browser"

In [4]:
# Set a fixed random seed so that the same random numbers
# are generated every time the code is executed
np.random.seed(23)

In [5]:
mu_vec1 = np.array([0,0,0])
cov_mat1 = np.array([[1,0,0], [0,1,0], [0,0,1]])
class1_sample = np.random.multivariate_normal(mu_vec1, cov_mat1, 20)

In [6]:
df = pd.DataFrame(class1_sample, columns=['feature1', 'feature2', 'feature3'])
df['target'] = 1

In [8]:
mu_vec2 = np.array([1,1,1])
cov_mat2 = np.array([[1,0,0], [0,1,0], [0,0,1]])
class2_sample = np.random.multivariate_normal(mu_vec2, cov_mat2, 20)

In [9]:
df1 = pd.DataFrame(class2_sample, columns=['feature1', 'feature2', 'feature3'])
df1['target'] = 0

In [17]:
df = pd.concat([df, df1], ignore_index=True)
df = df.sample(40)

In [18]:
df.head()

,feature1,feature2,feature3,target
25,0.290746,0.866975,0.982643,0
2,-0.367548,-1.137460,-1.322148,1
1,0.948634,0.701672,-1.051082,1
19,-0.992574,-0.161346,1.192404,1
11,1.968435,-0.547788,-0.679418,1


In [23]:
import plotly.express as px
#y_train_trf = y_train.astype(str)
fig = px.scatter_3d(df, x=df['feature1'], y=df['feature2'], z=df['feature3'],
              color=df['target'].astype('str'))
fig.update_traces(marker=dict(size=12,
                              line=dict(width=2,
                                        color='DarkSlateGrey')),
                  selector=dict(mode='markers'))

fig.show()

In [24]:
# Step 1 - Apply standard scaling
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

df.iloc[:,0:3] = scaler.fit_transform(df.iloc[:,0:3])

In [25]:
# Step 2 - Find Covariance Matrix
covariance_matrix = np.cov([df.iloc[:,0],df.iloc[:,1],df.iloc[:,2]])
print('Covariance Matrix:\n', covariance_matrix)

Covariance Matrix:
 [[1.02564103 0.18331431 0.00550246]
 [0.18331431 1.02564103 0.17466252]
 [0.00550246 0.17466252 1.02564103]]


In [26]:
# Step 3 - Finding EV and EVs
eigen_values, eigen_vectors = np.linalg.eig(covariance_matrix)

In [27]:
eigen_values

array([1.28160585, 1.020145  , 0.77517223])

In [28]:
eigen_vectors

array([[-0.51420659, -0.68979868,  0.50967575],
       [-0.70325903, -0.00105082, -0.71093293],
       [-0.49093617,  0.72400047,  0.48456681]])

In [ ]:
%pylab inline

# Ye Jupyter Notebook ka magic command hai jo:

# 👉 NumPy + Matplotlib ko import karta hai
# 👉 Aur plots ko notebook ke andar hi show karta hai (inline)

%pylab is deprecated, use %matplotlib inline and import the required libraries.
Populating the interactive namespace from numpy and matplotlib


In [ ]:
from matplotlib import pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from mpl_toolkits.mplot3d import proj3d
from matplotlib.patches import FancyArrowPatch

# pyplot → normal drawing (paper)
# Axes3D → 3D box drawing
# proj3d → camera projection
# FancyArrowPatch → arrows showing direction

In [35]:
class Arrow3D(FancyArrowPatch):
    def __init__(self, xs, ys, zs, *args, **kwargs):
        super().__init__((0,0), (0,0), *args, **kwargs)
        self._verts3d = xs, ys, zs

    def draw(self, renderer):
        xs3d, ys3d, zs3d = self._verts3d
        xs, ys, zs = proj3d.proj_transform(xs3d, ys3d, zs3d, renderer.M)
        self.set_positions((xs[0], ys[0]), (xs[1], ys[1]))
        super().draw(renderer)

    # 🔥 THIS FIXES YOUR ERROR
    def do_3d_projection(self, renderer=None):
        xs3d, ys3d, zs3d = self._verts3d
        xs, ys, zs = proj3d.proj_transform(xs3d, ys3d, zs3d, None)
        return min(zs)

In [36]:
pc = eigen_vectors[0:2]
pc

array([[-0.51420659, -0.68979868,  0.50967575],
       [-0.70325903, -0.00105082, -0.71093293]])

In [37]:
transformed_df = np.dot(df.iloc[:,0:3],pc.T)
# 40,3 - 3,2
new_df = pd.DataFrame(transformed_df,columns=['PC1','PC2'])
new_df['target'] = df['target'].values
new_df.head()

,PC1,PC2,target
0,0.006575,-0.068237,0
1,0.540095,1.883314,1
2,-1.160359,0.848464,1
3,1.388150,0.626870,1
4,-0.649893,-0.057457,1


In [38]:
new_df['target'] = new_df['target'].astype('str')
fig = px.scatter(x=new_df['PC1'],
                 y=new_df['PC2'],
                 color=new_df['target'],
                 color_discrete_sequence=px.colors.qualitative.G10
                )

fig.update_traces(marker=dict(size=12,
                              line=dict(width=2,
                                        color='DarkSlateGrey')),
                  selector=dict(mode='markers'))
fig.show()